In [3]:
require(data.table)
require(tidyverse)
require(dada2)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)
options(repr.plot.width=20, repr.plot.height=15)

Loading required package: metacoder

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘metacoder’”
Loading required package: DESeq2

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘DESeq2’”


In [2]:
asv1=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run1_16S_ASV_nochim.csv")

asv2=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run2_16S_ASV_nochim.csv")

asv3=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run3_16S_ASV_nochim.csv")

asv4=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run4_16S_ASV_nochim.csv")

asv5=fread("/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run5_16S_ASV_nochim.csv")

In [4]:
#make sure sample names are rownames
to_mat <- function(df) {
  df <- as.data.frame(df)
  rownames(df) <- df[,1]
  df <- df[,-1]
  as.matrix(df)
}

asv1 <- to_mat(asv1)
asv2 <- to_mat(asv2)
asv3 <- to_mat(asv3)
asv4 <- to_mat(asv4)
asv5 <- to_mat(asv5)

In [5]:
#merge using dada2

all_asvs <- Reduce(union, list(
  colnames(asv1),
  colnames(asv2),
  colnames(asv3),
  colnames(asv4),
  colnames(asv5)
))

merge_asv <- function(mat, all_cols) {
  missing <- setdiff(all_cols, colnames(mat))
  if (length(missing) > 0) {
    zero_mat <- matrix(0, nrow = nrow(mat), ncol = length(missing))
    colnames(zero_mat) <- missing
    mat <- cbind(mat, zero_mat)
  }
  mat[, all_cols, drop = FALSE]
}

asv_list <- lapply(list(asv1, asv2, asv3, asv4, asv5),
                   merge_asv,
                   all_cols = all_asvs)

seqtab.all <- do.call(rbind, asv_list)

In [6]:
#check 
sum(seqtab.all) #should be 167M reads
sum(is.na(seqtab.all)) #must be zero

[1] 170472134

[1] 0

In [7]:
nrow(seqtab.all) #475

[1] 484

## now work on taxa

In [ ]:
tax1=read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run1_16S_taxa.csv", header=TRUE, row.names=1)

tax2=read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run2_16S_taxa.csv", header=TRUE, row.names=1)

tax3=read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run3_16S_taxa.csv", header=TRUE, row.names=1)

tax4=read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run4_16S_taxa.csv", header=TRUE, row.names=1)

tax5=read.csv(file="/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/ASVs/Run5_16S_taxa.csv", header=TRUE, row.names=1)

In [ ]:
#make sure taxa tables are chr matrices
tax_list <- list(tax1, tax2, tax3, tax4, tax5)
tax_list <- lapply(tax_list, function(x) as.matrix(as.data.frame(x)))

In [ ]:
#build full asv

all_asvs <- Reduce(union, lapply(tax_list, rownames)) #all taxa id

In [ ]:
#align tax1 to tax2...

align_tax <- function(tax, all_asvs) {
  
  missing <- setdiff(all_asvs, rownames(tax))
  
  if (length(missing) > 0) {
    add <- matrix(NA,
                  nrow = length(missing),
                  ncol = ncol(tax))
    rownames(add) <- missing
    colnames(add) <- colnames(tax)
    tax <- rbind(tax, add)
  }
  
  tax[all_asvs, , drop = FALSE]
}

In [ ]:
#can also use this cleaner version to align
tax_list <- lapply(tax_list, function(x) {
  x <- as.matrix(x)
  x <- x[order(rownames(x)), ]
  x
})

In [8]:
#merge all runs
tax.all <- Reduce(function(x, y) align_tax(x, all_asvs), ## all sequence ids from taxonomy
                  tax_list)

In [ ]:
#check
any(duplicated(rownames(tax.all))) #no dupes

#full coverage
length(setdiff(all_asvs, rownames(tax.all))) #should be 0

#any missing
colSums(is.na(tax.all))

In [9]:
tax <- bind_rows(tax1, tax2, tax3, tax4, tax5)

ERROR: Error: object 'tax1' not found


In [ ]:
all_asvs <- union(rownames(tax1),
                  union(rownames(tax2),
                        union(rownames(tax3),
                              union(rownames(tax4), rownames(tax5)))))

In [ ]:
merge_tax <- function(tax, all_asvs) {
  missing <- setdiff(all_asvs, rownames(tax))
  if (length(missing) > 0) {
    add <- matrix(NA, nrow = length(missing), ncol = ncol(tax))
    rownames(add) <- missing
    colnames(add) <- colnames(tax)
    tax <- rbind(tax, add)
  }
  tax[all_asvs, , drop = FALSE]
}

In [ ]:
tax.all <- Reduce(function(x, y) merge_tax(x, all_asvs),
                  list(tax1, tax2, tax3, tax4, tax5))